In [1]:

with open("python_notes.txt", "w", encoding="utf-8") as file:
    file.write("Topic 1: Variables store data. Python is dynamically typed.\n")
    file.write("Topic 2: Lists are ordered and mutable.\n")
    file.write("Topic 3: Dictionaries store key-value pairs.\n")
    file.write("Topic 4: Loops automate repetitive tasks.\n")
    file.write("Topic 5: Exception handling prevents crashes.\n")

print("File written successfully.")

with open("python_notes.txt", "a", encoding="utf-8") as file:
    file.write("Topic 6: Functions help reuse code.\n")
    file.write("Topic 7: Modules help organize large programs.\n")

print("Lines appended.")

with open("python_notes.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

# Print numbered lines
print("\n--- File Content ---")
for i, line in enumerate(lines, start=1):
    print(f"{i}. {line.strip()}")

# Count total lines
total_lines = len(lines)
print(f"\nTotal number of lines: {total_lines}")

keyword = input("\nEnter a keyword to search: ").lower()
found = False
print("\n--- Matching Lines ---")
for line in lines:
    if keyword in line.lower():
        print(line.strip())
        found = True

if not found:
    print("No matching lines found.")

from google.colab import files
files.download("python_notes.txt")

File written successfully.
Lines appended.

--- File Content ---
1. Topic 1: Variables store data. Python is dynamically typed.
2. Topic 2: Lists are ordered and mutable.
3. Topic 3: Dictionaries store key-value pairs.
4. Topic 4: Loops automate repetitive tasks.
5. Topic 5: Exception handling prevents crashes.
6. Topic 6: Functions help reuse code.
7. Topic 7: Modules help organize large programs.

Total number of lines: 7

Enter a keyword to search: python

--- Matching Lines ---
Topic 1: Variables store data. Python is dynamically typed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Install requests (only needed in Colab first time)
!pip install requests

import requests
from datetime import datetime

# =========================
# LOGGER FUNCTION (Task 4)
# =========================
def log_error(context, error_type, message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open("error_log.txt", "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] ERROR in {context}: {error_type} — {message}\n")


# =========================
# SAFE REQUEST WRAPPERS (Task 3 Part C)
# =========================
def safe_get(url):
    try:
        response = requests.get(url, timeout=5)
        return response
    except requests.exceptions.ConnectionError:
        print("Connection failed. Please check your internet.")
        log_error("GET", "ConnectionError", "No connection")
    except requests.exceptions.Timeout:
        print("Request timed out. Try again later.")
        log_error("GET", "Timeout", "Request took too long")
    except Exception as e:
        print("Error:", e)
        log_error("GET", "Exception", str(e))
    return None


def safe_post(url, data):
    try:
        response = requests.post(url, json=data, timeout=5)
        return response
    except requests.exceptions.ConnectionError:
        print("Connection failed. Please check your internet.")
        log_error("POST", "ConnectionError", "No connection")
    except requests.exceptions.Timeout:
        print("Request timed out.")
        log_error("POST", "Timeout", "Request took too long")
    except Exception as e:
        print("Error:", e)
        log_error("POST", "Exception", str(e))
    return None


# =========================
# TASK 2 — STEP 1
# =========================
print("\n--- Fetching Products ---")

url = "https://dummyjson.com/products?limit=20"
response = safe_get(url)

products = []

if response and response.status_code == 200:
    data = response.json()
    products = data["products"]

    print(f"{'ID':<5} {'Title':<30} {'Category':<15} {'Price':<10} {'Rating'}")
    print("-"*75)

    for p in products:
        print(f"{p['id']:<5} {p['title'][:28]:<30} {p['category']:<15} ${p['price']:<10} {p['rating']}")
else:
    log_error("fetch_products", "HTTPError", f"Status code {response.status_code if response else 'None'}")


# =========================
# TASK 2 — STEP 2
# =========================
print("\n--- Filtered (Rating ≥ 4.5) & Sorted by Price ---")

filtered = [p for p in products if p["rating"] >= 4.5]
filtered_sorted = sorted(filtered, key=lambda x: x["price"], reverse=True)

for p in filtered_sorted:
    print(f"{p['title']} - ${p['price']} - Rating: {p['rating']}")


# =========================
# TASK 2 — STEP 3
# =========================
print("\n--- Laptop Products ---")

url = "https://dummyjson.com/products/category/laptops"
response = safe_get(url)

if response and response.status_code == 200:
    laptops = response.json()["products"]
    for l in laptops:
        print(f"{l['title']} - ${l['price']}")
else:
    log_error("fetch_laptops", "HTTPError", "Failed to fetch laptops")


# =========================
# TASK 2 — STEP 4 (POST)
# =========================
print("\n--- Creating Product (POST) ---")

post_data = {
    "title": "My Custom Product",
    "price": 999,
    "category": "electronics",
    "description": "A product I created via API"
}

response = safe_post("https://dummyjson.com/products/add", post_data)

if response:
    print(response.json())


# =========================
# TASK 3 — PART A
# =========================
print("\n--- Safe Divide ---")

def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "Error: Cannot divide by zero"
    except TypeError:
        return "Error: Invalid input types"

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide("ten", 2))


# =========================
# TASK 3 — PART B
# =========================
print("\n--- Safe File Read ---")

def read_file_safe(filename):
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
    finally:
        print("File operation attempt complete.")

print(read_file_safe("python_notes.txt"))
print(read_file_safe("ghost_file.txt"))


# =========================
# TASK 3 — PART D
# =========================
print("\n--- Product Lookup ---")

while True:
    user_input = input("Enter product ID (1–100) or 'quit': ")

    if user_input.lower() == "quit":
        break

    if not user_input.isdigit():
        print("Invalid input. Enter a number.")
        continue

    pid = int(user_input)
    if pid < 1 or pid > 100:
        print("Out of range. Try again.")
        continue

    response = safe_get(f"https://dummyjson.com/products/{pid}")

    if response:
        if response.status_code == 200:
            p = response.json()
            print(f"{p['title']} - ${p['price']}")
        elif response.status_code == 404:
            print("Product not found.")
            log_error("lookup_product", "HTTPError", f"404 for ID {pid}")


# =========================
# TASK 4 — FORCE ERRORS
# =========================
print("\n--- Triggering Errors ---")

# Fake URL → ConnectionError
safe_get("https://this-host-does-not-exist-xyz.com/api")

# Invalid product → HTTP 404
response = safe_get("https://dummyjson.com/products/999")
if response and response.status_code != 200:
    log_error("lookup_product", "HTTPError", "404 Not Found for product ID 999")


# =========================
# SHOW LOG FILE
# =========================
print("\n--- Error Log ---")

try:
    with open("error_log.txt", "r", encoding="utf-8") as f:
        print(f.read())
except:
    print("No logs found.")


--- Fetching Products ---
ID    Title                          Category        Price      Rating
---------------------------------------------------------------------------
1     Essence Mascara Lash Princes   beauty          $9.99       2.56
2     Eyeshadow Palette with Mirro   beauty          $19.99      2.86
3     Powder Canister                beauty          $14.99      4.64
4     Red Lipstick                   beauty          $12.99      4.36
5     Red Nail Polish                beauty          $8.99       4.32
6     Calvin Klein CK One            fragrances      $49.99      4.37
7     Chanel Coco Noir Eau De        fragrances      $129.99     4.26
8     Dior J'adore                   fragrances      $89.99      3.8
9     Dolce Shine Eau de             fragrances      $69.99      3.96
10    Gucci Bloom Eau de             fragrances      $79.99      2.74
11    Annibale Colombo Bed           furniture       $1899.99    4.77
12    Annibale Colombo Sofa          furniture       $249